In [ ]:
# --- paths come from human/config.py (auto-inserted by fix_notebooks.py) ---
import sys; sys.path.append('..')
from config import HUMAN_BASE


# BMMC SETIA Input — State Calling using Yeast Pipeline

This notebook applies the yeast `GRN_input_acquisition.py` state-calling pipeline **verbatim** to BMMC data.

**Strategy**:
1. Preprocess BMMC scRNA-seq → pseudocells (each cell type → K pseudocells, each = aggregate of ~30+ raw cells)
2. Each pseudocell plays the role of a yeast biological replicate
3. Each cell type plays the role of a yeast TF-KO condition
4. Call yeast's `select_and_convert_gmm_aicc()` unchanged for each of 86 genes
5. Output SETIA input matrix (cell_type × gene, values = state medians)

**Pseudocell strategy**: `K = min(10, n_cells // 30)`, with at least 2 pseudocells per cell type.

---
## Part A: Imports and yeast state-calling functions (verbatim)

In [ ]:
import os
import math
import csv
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from itertools import combinations
from scipy.stats import ttest_ind, gaussian_kde
from statsmodels.stats.multitest import multipletests
import anndata as ad
from scipy.sparse import issparse, csr_matrix

np.seterr(all='ignore')

# Module-level constant replacing args.p_value_cutoff in the yeast script.
# Default in yeast pipeline was 0.05.
P_VALUE_CUTOFF = 0.01

# Output folders (yeast pipeline writes intermediate files here)
os.makedirs(f"{HUMAN_BASE}/GTEx_v11/pseudo/result", exist_ok=True)
os.makedirs(f"{HUMAN_BASE}/GTEx_v11/pseudo/result/GMM_figures/AIC", exist_ok=True)

# Clean up output log so it doesn't accumulate across runs
_STATE_COUNT_FILE = f"{HUMAN_BASE}/GTEx_v11/pseudo/result/Steady_state_count.txt"
if os.path.exists(_STATE_COUNT_FILE):
    os.remove(_STATE_COUNT_FILE)

print('Setup OK')

### A1. `kde_likelihood_empirical_p` — verbatim from yeast pipeline

In [ ]:
def kde_likelihood_empirical_p(A, B, bw_method=None, n_permutations=2000,
                               alternative='greater', eps=1e-300,
                               random_seed=None, return_details=False):
    """
    Compute KDE-based geometric-mean score for B under KDE(A) and convert it to an
    empirical p-value via permutation.
    """
    if random_seed is not None:
        np.random.seed(random_seed)

    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    if n_permutations < 1:
        raise ValueError("n_permutations must be >= 1")

    nA = len(A)
    nB = len(B)
    if nA < 2:
        raise ValueError("A must contain at least 2 points for KDE")

    def log_geo_mean_for_split(Atrain, Btest):
        kde = gaussian_kde(Atrain, bw_method=bw_method)
        dens = kde(Btest)
        logdens = np.log(dens + eps)
        return float(np.mean(logdens)), np.exp(np.mean(logdens))

    obs_loggm, obs_gm = log_geo_mean_for_split(A, B)

    combined = np.concatenate([A, B])
    perm_loggms = np.empty(n_permutations, dtype=float)
    for i in range(n_permutations):
        perm = np.random.permutation(combined)
        Aperm = perm[:nA]
        Bperm = perm[nA:]
        lgm, _ = log_geo_mean_for_split(Aperm, Bperm)
        perm_loggms[i] = lgm

    if alternative == 'greater':
        p_emp = (np.sum(perm_loggms >= obs_loggm) + 1) / (n_permutations + 1)
    elif alternative == 'less':
        p_emp = (np.sum(perm_loggms <= obs_loggm) + 1) / (n_permutations + 1)
    elif alternative == 'two-sided':
        greater = (np.sum(perm_loggms >= obs_loggm) + 1) / (n_permutations + 1)
        less    = (np.sum(perm_loggms <= obs_loggm) + 1) / (n_permutations + 1)
        p_emp = 2.0 * min(greater, less)
        p_emp = min(p_emp, 1.0)
    else:
        raise ValueError("alternative must be 'greater', 'less', or 'two-sided'")

    if return_details:
        return {
            'p_emp': float(p_emp),
            'obs_loggm': float(obs_loggm),
            'obs_gm': float(obs_gm),
            'perm_loggms': perm_loggms,
            'perm_gms': np.exp(perm_loggms)
        }
    return float(1-p_emp)

### A2. `teset_and_merge_welch` — verbatim from yeast pipeline (only `args.p_value_cutoff` → `P_VALUE_CUTOFF`)

In [ ]:
def teset_and_merge_welch(optimal_clusters, optimal_mapping, replicate_variability=None, alpha=P_VALUE_CUTOFF):
    """
    Merge clusters greedily using Welch's t-test (verbatim from yeast pipeline).
    """
    def avg_pairwise_dist(sub):
        arr = np.asarray(sub, dtype=float)
        arr = arr[~np.isnan(arr)]
        if arr.size < 2:
            return 0.0
        diffs = np.abs(arr[:, None] - arr)
        triu = diffs[np.triu_indices(arr.size, k=1)]
        return float(np.mean(triu))

    merged = [list(sub) for sub in optimal_clusters]

    try:
        merged_map = [list(optimal_mapping[i]) for i in range(len(optimal_clusters))]
    except Exception:
        keys_sorted = sorted(optimal_mapping.keys())
        merged_map = [list(optimal_mapping[k]) for k in keys_sorted]
        if len(merged_map) < len(merged):
            start = max(keys_sorted) + 1 if keys_sorted else 0
            for idx in range(len(merged_map), len(merged)):
                merged_map.append([start + (idx - len(merged_map))])

    within_dists = [avg_pairwise_dist(sub) for sub in merged]
    mean_within = float(np.mean(within_dists)) if within_dists else 0.0

    while True:
        n = len(merged)
        if n <= 1:
            break

        p_mat = np.full((n, n), -np.inf, dtype=float)

        for i in range(n):
            for j in range(i + 1, n):
                a = np.asarray(merged[i], dtype=float)
                b = np.asarray(merged[j], dtype=float)
                a = a[~np.isnan(a)]
                b = b[~np.isnan(b)]

                if a.size < 2 or b.size < 2:
                    continue

                try:
                    t_stat, p_val = ttest_ind(a, b, equal_var=False)
                    if len(a) < len(b):
                        gm = kde_likelihood_empirical_p(b, a, bw_method='scott', n_permutations=2000, alternative='greater', random_seed=0)
                    else:
                        gm = kde_likelihood_empirical_p(a, b, bw_method='scott', n_permutations=2000, alternative='greater', random_seed=0)
                    p_val = max(p_val, gm)
                except Exception:
                    continue

                if np.isnan(p_val):
                    continue

                p_mat[i, j] = p_val
                p_mat[j, i] = p_val

        max_p = np.max(p_mat)
        if not np.isfinite(max_p) or max_p < alpha:
            break

        flat_idx = np.argmax(p_mat)
        i, j = divmod(flat_idx, n)
        if i == j:
            break
        if i > j:
            i, j = j, i

        merged[i].extend(merged[j])
        merged_map[i].extend(merged_map[j])
        del merged[j]
        del merged_map[j]

    merged_mapping = {k: merged_map[k] for k in range(len(merged_map))}
    return merged, merged_mapping

### A3. Helper functions — verbatim from yeast pipeline

In [ ]:
def find_elbow_idx_by_cutoff(seq, cutoff):
    if len(seq) < 2:
        return None

    freq = Counter(seq)
    most_val = max(freq.items(), key=lambda kv: (kv[1], kv[0]))[0]

    drops = [seq[i] - seq[i + 1] for i in range(len(seq) - 1)]

    pos_sizes = sorted({d for d in drops if d > 0}, reverse=True)
    if not pos_sizes:
        return most_val, None

    for size in pos_sizes:
        for i, d in enumerate(drops):
            if d == size:
                after_val = seq[i + 1]
                if after_val <= cutoff:
                    return most_val, i + 1

def composite_score(x, y):
    if len(x) == 0 or len(y) == 0:
        return math.nan
    delta = abs(cliff_delta(x, y))
    hl    = abs(hodges_lehmann(x, y))
    return delta * hl

def hodges_lehmann(x, y):
    diffs = [xi - yj for xi in x for yj in y]
    return abs(np.median(diffs))

def cliff_delta(g1, g2):
    nx, ny = len(g1), len(g2)
    greater = sum(x > y for x in g1 for y in g2)
    less    = sum(x < y for x in g1 for y in g2)
    delta = (greater - less) / (nx*ny)
    return abs(delta)

def safe_kde(x, **kwargs):
    x = np.asarray(x)
    if x.size < 2 or np.all(x == x.flat[0]):
        return None
    return gaussian_kde(x, **kwargs)

def compute_all_ad_pairs(groups):
    records = []
    for (i, g1), (j, g2) in combinations(enumerate(groups), 2):
        p = composite_score(g1, g2)
        records.append({'group1': i, 'group2': j, 'p_value': p})
    return pd.DataFrame(records)

def merge_with_threshold(groups, df_pairs, alpha):
    n = len(groups)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for _, row in df_pairs.iterrows():
        if row.p_value <= alpha:
            union(int(row.group1), int(row.group2))

    comps = defaultdict(list)
    for i in range(n):
        comps[find(i)].append(i)

    merged = []
    merged_map = {}

    for merged_idx, idxs in enumerate(comps.values()):
        buf = []
        for idx in idxs:
            buf.extend(groups[idx])
        merged.append(buf)
        merged_map[merged_idx] = idxs

    return merged, merged_map

def assign_component(x, groups, mapping, original_comp):
    if any(set(sublist) == set(x) for sublist in original_comp) and len(x) > 1:
        for keys in mapping:
            if [i for i, sublist in enumerate(original_comp) if Counter(sublist) == Counter(x)][0] in mapping[keys]:
                return keys
            else:
                pass
    else:
        distance_x_avg = []
        for each_subgroup in groups:
            distance_x_avg.append(abs(np.mean(each_subgroup) - np.mean(x)))
        idx_best = distance_x_avg.index(min(distance_x_avg))
    return idx_best

def get_sample_index_in_TPM_list(Samples_Dic, TPM_values_for_gene, Factor):
    sample_index = 0
    for i in range(0, sorted(Samples_Dic.keys()).index(Factor)):
        sample_index = sample_index + len(TPM_values_for_gene[i])
    sample_span = [sample_index, sample_index+len(TPM_values_for_gene[sorted(Samples_Dic.keys()).index(Factor)])]
    return sample_span

### A4. `plot_GMM_distribution` — verbatim

In [ ]:
def plot_GMM_distribution(values_, batch_labels, merged_clusters_, mapping_, original_comp_, best_n_, replicate_index, max_p_value, Factor_name='', outname='GMM'):
    fig, axes = plt.subplots(2, 1, figsize=(9, 10), sharex=True)
    Hexcode_colors = ['#FF0000', '#FF7F00', '#FFFF00', '#00FF00', '#0000FF', '#4B0082', '#8B00FF', '#FF00FF']
    axes[0].hist(values_, bins=100, density=True, alpha=0.3, color='gray')
    for cluster_i, cluster_data in enumerate(merged_clusters_):
        cluster_data = np.array(cluster_data)
        kde = safe_kde(cluster_data)
        if kde is None:
            axes[0].axhline(cluster_data.flat[0], color=Hexcode_colors[cluster_i % len(Hexcode_colors)])
        else:
            xs = np.linspace(cluster_data.min(), cluster_data.max(), 2*len(cluster_data))
            weight = len(cluster_data)/len(values_)
            axes[0].plot(xs, weight*kde(xs), color=Hexcode_colors[cluster_i % len(Hexcode_colors)])
    axes[0].text(0.95, 0.95, "Histogram of {} samples\nMax ttest pvalue: {}".format(len(values_), format(float(max_p_value), ".2e")), horizontalalignment="right", verticalalignment="top", transform=axes[0].transAxes, fontsize=10, color="black")
    axes[0].tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    axes[0].set_ylabel("Normalized sample freq")
    axes[0].set_ylim()
    axes[0].set_title(fr"Bottom Up Model Fit for $\it{{{Factor_name}}}$ (n_component={best_n_})")
    df_combined = []
    df_all = []
    df_mean = []
    if len(replicate_index) > 0:
        for key in replicate_index:
            safe_key = key.replace('_', r'\_')
            x_scatter = []
            batch_scatter = []
            for j in range(replicate_index[key][0], replicate_index[key][1]):
                x_scatter.append(values_[j])
                batch_scatter.append(batch_labels[j])
            if len(x_scatter) != 0:
                df_temp = pd.DataFrame({
                    'value': x_scatter,
                    'batch' : batch_scatter,
                    'group': [fr"$\it{{{safe_key}}}$"] * len(x_scatter)
                })
                df_all.append(df_temp)
                df_mean.append(np.mean(x_scatter))
            else:
                pass
        df_all = [b for _, b in sorted(zip(df_mean, df_all), key=lambda x: x[0], reverse=False)]
    else:
        pass
    df_combined = pd.concat(df_all) if df_all else pd.DataFrame()
    unique_groups = df_combined['group'].unique() if len(df_combined) else []

    for each_df_i in range(0, len(df_all)):
        component_label_ = assign_component(df_all[each_df_i]['value'].tolist(), merged_clusters_, mapping_, original_comp_)
    for i, group in enumerate(unique_groups):
        subset = df_combined[df_combined['group'] == group]
        y_pos = np.full(len(subset), i + 1)
        component_label_ = assign_component(subset['value'].tolist(), merged_clusters_, mapping_, original_comp_)
        axes[1].scatter(subset['value'], y_pos, color=Hexcode_colors[component_label_ % len(Hexcode_colors)], s=45, alpha=1)
        axes[1].hlines(y=y_pos, xmin=min(subset['value']), xmax=max(subset['value']), colors=Hexcode_colors[component_label_ % len(Hexcode_colors)], linestyles='-', linewidth=0.5)

    axes[1].tick_params(axis='y', which='both', left=False, labelleft=False)
    axes[1].set_ylabel("Normalized sample freq")
    axes[1].set_xlabel("CPM")
    axes[1].set_ylabel("Cell type (pseudocells shown)")
    plt.tight_layout()
    os.makedirs('./result/GMM_figures/AIC', exist_ok=True)
    plt.savefig('./result/GMM_figures/AIC/{}_{}.jpg'.format(Factor_name, outname), dpi=300)
    plt.close()
    return

### A5. `select_and_convert_gmm_aicc` — the core state-calling function, verbatim

In [ ]:
def select_and_convert_gmm_aicc(Samples_Dic, TPM_values_for_gene, Batch_values_for_gene, Factor_name_, max_components=8):
    values = [x for sublist in TPM_values_for_gene for x in sublist]
    batch_labels = [x for sublist in Batch_values_for_gene for x in sublist]

    ######################################## Calculate the variability between replicates ########################################
    replicate_variability = []
    for each_replicate_values in TPM_values_for_gene:
        replicate_variability.append(0 if len(each_replicate_values) < 2 else sum(abs(x-y) for i,x in enumerate(each_replicate_values) for y in each_replicate_values[i+1:]) / (len(each_replicate_values)*(len(each_replicate_values)-1)/2))
    ######################################## Calculate the variability between replicates ########################################

    ########################################## Use elbow method on the Cliff delta ###############################################
    clustering_sensitivity = 100
    TPM_values_for_gene_cleaned = [vector for vector in TPM_values_for_gene if len(vector) > 1]
    df_pairs = compute_all_ad_pairs(TPM_values_for_gene_cleaned)

    number_of_clusters = [len(merge_with_threshold(TPM_values_for_gene_cleaned, df_pairs, test_alpha)[0]) for test_alpha in np.linspace(df_pairs['p_value'].min(), df_pairs['p_value'].max(), clustering_sensitivity, endpoint=False)]
    most_val, idx_of_elbow = find_elbow_idx_by_cutoff(number_of_clusters, 8)
    if idx_of_elbow == None:
        optimal_alpha = df_pairs['p_value'].min()
    elif most_val == 1:
        if number_of_clusters.count(1) >= len(number_of_clusters)-1:
            optimal_alpha = np.linspace(df_pairs['p_value'].min(), df_pairs['p_value'].max(), clustering_sensitivity, endpoint=False)[number_of_clusters.index(1)]
        else:
            optimal_alpha = np.linspace(df_pairs['p_value'].min(), df_pairs['p_value'].max(), clustering_sensitivity, endpoint=False)[idx_of_elbow]
    else:
        optimal_alpha = np.linspace(df_pairs['p_value'].min(), df_pairs['p_value'].max(), clustering_sensitivity, endpoint=False)[idx_of_elbow]
    optimal_clusters, optimal_mapping = merge_with_threshold(TPM_values_for_gene_cleaned, df_pairs, optimal_alpha)
    filtered_clusters = []
    filtered_mapping = {}
    for new_idx, old_idx in enumerate(range(len(optimal_clusters))):
        cluster = optimal_clusters[old_idx]
        if len(cluster) > 1:
            filtered_clusters.append(cluster)
            filtered_mapping[len(filtered_clusters)-1] = optimal_mapping[old_idx]
    optimal_clusters = filtered_clusters
    optimal_mapping = {new_idx: filtered_mapping[old_key] for new_idx, old_key in enumerate(sorted(filtered_mapping.keys()))}
    optimal_clusters, optimal_mapping = teset_and_merge_welch(optimal_clusters, optimal_mapping, sum(sorted(replicate_variability)[-3:]) / 3)
    optimal_mapping = {new_idx: optimal_mapping[old_key] for new_idx, old_key in enumerate(sorted(optimal_mapping.keys()))}
    ########################################## Use elbow method on the Cliff delta ###############################################

    best_n = len(optimal_clusters)
    medians = [np.median(c) for c in optimal_clusters]
    means = [np.mean(c) for c in optimal_clusters]
    covs     = [np.std(c)    for c in optimal_clusters]
    # Also compute linear-space medians and stds from the SAME clusters,
    # by transforming each cluster back to linear (2^x - 1) before stats.
    # This lets us output both log2 and linear stats from one GMM run.
    linear_clusters = [(2.0 ** np.array(c)) - 1.0 for c in optimal_clusters]
    linear_clusters = [np.clip(c, 0, None) for c in linear_clusters]
    medians_linear = [float(np.median(c)) for c in linear_clusters]
    covs_linear    = [float(np.std(c))    for c in linear_clusters]
    weights  = [len(c)/len(values) for c in optimal_clusters]
    order = np.argsort(means)
    optimal_clusters = [optimal_clusters[i] for i in order]
    optimal_mapping = {new_idx: optimal_mapping[old_idx] for new_idx, old_idx in enumerate(order)}
    medians = [medians[i]          for i in order]
    covs = [covs[i]             for i in order]
    weights = [weights[i]          for i in order]
    means = [means[i]          for i in order]
    medians_linear = [medians_linear[i] for i in order]
    covs_linear    = [covs_linear[i]    for i in order]
    outfile = open('./result/Steady_state_count.txt', 'a')
    outfile.write(Factor_name_+'\t'+str(best_n)+'\t')
    converted_GMM_TPM = []
    converted_GMM_TPM_linear = []
    converted_GMM_std = []
    converted_GMM_std_linear = []
    for idx, each_TPMs in enumerate(TPM_values_for_gene):
        if each_TPMs == []:
            converted_GMM_TPM.append([0])
            converted_GMM_TPM_linear.append([0])
            converted_GMM_std.append([0])
            converted_GMM_std_linear.append([0])
        else:
            component_label = assign_component(each_TPMs, optimal_clusters, optimal_mapping, TPM_values_for_gene_cleaned)
            converted_GMM_TPM.append(medians[component_label])
            converted_GMM_TPM_linear.append(medians_linear[component_label])
            converted_GMM_std.append(covs[component_label])
            converted_GMM_std_linear.append(covs_linear[component_label])
            outfile.write(sorted(Samples_Dic.keys())[idx] + ':' + str(component_label) + '\t')
    outfile.write('\n')
    replicate_index = {}
    replicates_to_plot = list(Samples_Dic.keys())
    for each in replicates_to_plot:
        replicate_index[each] = get_sample_index_in_TPM_list(Samples_Dic, TPM_values_for_gene, each)

    pairs = list(itertools.combinations(range(len(optimal_clusters)), 2))
    pvals = []
    results = []

    replicate_test_p = []
    if len(optimal_clusters) > 1:
        for i, j in pairs:
            t_stat, p_val = ttest_ind(optimal_clusters[i], optimal_clusters[j], equal_var=False)
            pvals.append(p_val)
            results.append((str(i), str(j), t_stat, p_val))

        reject, pvals_corr, _, _ = multipletests(pvals, method="holm")

        for (g1, g2, t, p), p_corr, r in zip(results, pvals_corr, reject):
            replicate_test_p.append(p_corr)
    else:
        pass
    outfile.close()

    plot_GMM_distribution(np.array(values), batch_labels, optimal_clusters, optimal_mapping, TPM_values_for_gene_cleaned, best_n, replicate_index, float(max(replicate_test_p)) if replicate_test_p else math.nan, Factor_name=Factor_name_)

    for each_i in range(0, len(converted_GMM_TPM)):
        if isinstance(converted_GMM_TPM[each_i], list):
            converted_GMM_TPM[each_i] = converted_GMM_TPM[each_i][0]
            if isinstance(converted_GMM_TPM_linear[each_i], list):
                converted_GMM_TPM_linear[each_i] = converted_GMM_TPM_linear[each_i][0]
            if isinstance(converted_GMM_std_linear[each_i], list):
                converted_GMM_std_linear[each_i] = converted_GMM_std_linear[each_i][0]
            converted_GMM_std[each_i] = converted_GMM_std[each_i][0]
        elif isinstance(converted_GMM_TPM[each_i], np.ndarray):
            converted_GMM_TPM[each_i] = converted_GMM_TPM[each_i].item()
            converted_GMM_std[each_i] = converted_GMM_std[each_i].item()
        else:
            pass

    return (best_n,
            np.array(converted_GMM_TPM),        np.array(converted_GMM_std),
            np.array(converted_GMM_TPM_linear), np.array(converted_GMM_std_linear))

print('All yeast state-calling functions loaded.')

---
## Part B: BMMC preprocessing — build `Samples_Dic` and `mRNA_steady_states`

**Goal**: produce data structures that look exactly like what the yeast pipeline expects.

Yeast format:
- `Samples_Dic[condition] = [replicate_id_1, replicate_id_2, ...]`  
- `mRNA_steady_states[replicate_id][gene] = TPM value`

BMMC mapping:
- `condition` ↔ cell type (e.g., `'01_HSC'`)
- `replicate_id` ↔ pseudocell (e.g., `'01_HSC_pseudo_001'`)
- `TPM value` ↔ CPM value

In [ ]:
GENE_LIST_PATH = f"{HUMAN_BASE}/bmmc_gene_list.tsv"
H5AD_PATH      = f"{HUMAN_BASE}/GTEx_v11/Granja2019_annotated.h5ad"
CELLTYPE_COL   = 'BioClassification'

# Pseudocell parameters
MAX_PSEUDOCELLS_PER_CELLTYPE = 10
MIN_CELLS_PER_PSEUDOCELL     = 50
MIN_PSEUDOCELLS              = 2   # state calling needs at least 2 pseudocells per condition
RANDOM_SEED                  = 42

print(f'pseudocell params: max={MAX_PSEUDOCELLS_PER_CELLTYPE}, min_cells_per={MIN_CELLS_PER_PSEUDOCELL}')

### B1. Load gene list (canonical order)

In [ ]:
gene_list_df = pd.read_csv(GENE_LIST_PATH, sep='\t', comment='#')
gene_list_df
ordered_genes = gene_list_df['gene_symbol'].tolist()
print(f'Genes in list (in order): {len(ordered_genes)}')
print(f'  First 5: {ordered_genes[:5]}')
print(f'  Last 5:  {ordered_genes[-5:]}')

### B2. Load AnnData, grab raw counts, drop genes not in list

In [ ]:
adata = ad.read_h5ad(H5AD_PATH)
X_raw = adata.raw.X
gene_names = pd.Index(adata.raw.var_names)

if not issparse(X_raw):
    X_raw = csr_matrix(X_raw)
if np.isnan(X_raw.data).any():
    print(f'WARN: replacing {np.isnan(X_raw.data).sum()} NaN raw count values with 0')
    X_raw.data = np.nan_to_num(X_raw.data, nan=0.0)

# Subset to only the 86 genes we care about (keeps everything fast)
missing = [g for g in ordered_genes if g not in gene_names]
if missing:
    raise RuntimeError(f'Genes missing from raw.var_names: {missing}')

gene_idx = [gene_names.get_loc(g) for g in ordered_genes]
X_raw_sub = X_raw[:, gene_idx]
print(f'Raw counts subset: {X_raw_sub.shape} (cells × {len(ordered_genes)} genes)')
print(f'Total UMI per cell (using ALL genes, not just our 86):')
lib_size_all = np.asarray(X_raw.sum(axis=1)).flatten()
print(f'  median={np.median(lib_size_all):.0f}, mean={np.mean(lib_size_all):.0f}')

### B3. Generate pseudocells per cell type and compute CPM

For each cell type:
1. Shuffle the cells (seeded)
2. Split into `K = min(MAX, n_cells // MIN_CELLS_PER_PSEUDOCELL)` groups, `K >= MIN_PSEUDOCELLS`
3. Sum raw counts within each group
4. CPM normalize using **total library size of the pseudocell** (across all 13460 genes, not just our 86)

**Why CPM with total library**: matches what you'd get if you treated this pseudocell as a bulk RNA-seq sample. Using only the 86-gene total would over-inflate CPM for highly-expressed marker genes.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
celltypes = adata.obs[CELLTYPE_COL].astype(str).values
unique_cts = sorted(set(celltypes))

Samples_Dic         = {}   # cell_type -> [pseudocell_id_1, ...]
mRNA_steady_states  = {}   # pseudocell_id -> {gene: CPM}

pseudocell_info = []
for ct in unique_cts:
    cell_mask = np.where(celltypes == ct)[0]
    n_cells = len(cell_mask)
    if n_cells < MIN_CELLS_PER_PSEUDOCELL * MIN_PSEUDOCELLS:
        # Need at least 60 cells to make 2 pseudocells of 30+ cells
        # Try to make 2 pseudocells anyway if we have enough
        if n_cells >= 2 * 10:  # absolute fallback: >=20 cells
            K = MIN_PSEUDOCELLS
        else:
            print(f'  SKIP {ct}: only {n_cells} cells (need at least 20 for 2 pseudocells of 10)')
            continue
    else:
        K = min(MAX_PSEUDOCELLS_PER_CELLTYPE, n_cells // MIN_CELLS_PER_PSEUDOCELL)
    K = max(K, MIN_PSEUDOCELLS)
    
    shuffled = rng.permutation(cell_mask)
    chunks = np.array_split(shuffled, K)
    
    pseudocell_ids = []
    for i, chunk in enumerate(chunks):
        pseudocell_id = f'{ct}_pseudo_{i:03d}'
        
        # Sum raw counts over cells in chunk (using ALL genes for proper library size)
        # For our 86 genes:
        chunk_counts_86 = np.asarray(X_raw_sub[chunk].sum(axis=0)).flatten()
        # Total library size:
        chunk_lib_size = float(np.asarray(X_raw[chunk].sum()).flatten()[0])
        
        if chunk_lib_size <= 0:
            print(f'  WARN: {pseudocell_id} has zero library size, skipping')
            continue
        
        cpm = chunk_counts_86 / chunk_lib_size * 1e6
        
        mRNA_steady_states[pseudocell_id] = {
            gene: float(cpm[gi]) for gi, gene in enumerate(ordered_genes)
        }
        pseudocell_ids.append(pseudocell_id)
        pseudocell_info.append({
            'cell_type': ct, 'pseudocell': pseudocell_id,
            'n_cells': len(chunk), 'lib_size': chunk_lib_size,
        })
    
    Samples_Dic[ct] = pseudocell_ids

info_df = pd.DataFrame(pseudocell_info)
print(f'\nBuilt {len(mRNA_steady_states)} pseudocells across {len(Samples_Dic)} cell types')
print(f'Pseudocells per cell type:')
print(info_df.groupby('cell_type').size().to_string())

In [ ]:
print('Cells per pseudocell distribution:')
print(info_df['n_cells'].describe())
print(f'\nLib size per pseudocell distribution:')
print(info_df['lib_size'].describe())

---
## Part C: Run state calling for each gene

Exactly mirroring the yeast pipeline's main loop. For each gene, build `TPM_values_for_gene` as list-of-lists (one inner list per cell type), then call `select_and_convert_gmm_aicc`.

**Note**: log2-transform is applied to match yeast pipeline's convention (yeast uses `log2(TPM+1)` then state calling, then medians remain in log space — same here).

In [ ]:
# Apply log2(CPM + 1) transform to match yeast pipeline scale
# (yeast pipeline does this implicitly; CPM values pre-transform are 0-thousands)
USE_LOG2 = True

# Build a helper to look up gene values quickly
def gene_value(pseudocell_id, gene):
    v = mRNA_steady_states[pseudocell_id].get(gene, 0.0)
    if USE_LOG2:
        return float(np.log2(v + 1))
    return float(v)

In [ ]:
# Loop matches yeast pipeline lines 792-825 exactly
Column_order = ordered_genes

TPM_matrix_T = []
std_matrix_T = []
TPM_matrix_T_linear = []
std_matrix_T_linear = []
n_states_dict = {}

# Header row: gene index in Column_order for each cell type (or -1 if cell type name not in gene list)
row_header = []
for each_sample in sorted(Samples_Dic.keys()):
    if each_sample in Column_order:
        row_header.append(Column_order.index(each_sample))
    else:
        row_header.append(-1)
TPM_matrix_T.append(row_header)
std_matrix_T.append(row_header)
TPM_matrix_T_linear.append(row_header)
std_matrix_T_linear.append(row_header)

import time
t0 = time.time()

for col_idx, each_column in enumerate(Column_order):
    print(col_idx)
    TPM_values_for_gene = []
    Batch_values_for_gene = []
    for each_sample in sorted(Samples_Dic.keys()):
        TPM_values_for_replicates = []
        Batch_values_for_replicates = []
        # Yeast pipeline: skip if cell type name == gene name (self-knockout case)
        # In BMMC this never triggers because cell type names like '01_HSC' won't match gene names
        if each_column == each_sample:
            pass
        else:
            for each_replicate in Samples_Dic[each_sample]:
                TPM_values_for_replicates.append(gene_value(each_replicate, each_column))
                # Yeast uses each_replicate.split('_')[2] as batch label.
                # BMMC pseudocell id is e.g. '01_HSC_pseudo_001' -> split('_')[2] = 'pseudo'.
                # We use the same logic for compatibility with plot_GMM_distribution.
                Batch_values_for_replicates.append(each_replicate.split('_')[2] if len(each_replicate.split('_')) > 2 else 'b1')
        TPM_values_for_gene.append(TPM_values_for_replicates)
        Batch_values_for_gene.append(Batch_values_for_replicates)
    
    best_n, convertion_Dic, std_Dic, convertion_Dic_linear, std_Dic_linear = select_and_convert_gmm_aicc(
        Samples_Dic, TPM_values_for_gene, Batch_values_for_gene, each_column
    )
    
    TPM_matrix_T.append(list(convertion_Dic))
    std_matrix_T.append(list(std_Dic))
    TPM_matrix_T_linear.append(list(convertion_Dic_linear))
    std_matrix_T_linear.append(list(std_Dic_linear))
    n_states_dict[each_column] = best_n
    
    if (col_idx + 1) % 10 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (col_idx+1) * (len(Column_order) - col_idx - 1)
        print(f'  [{col_idx+1}/{len(Column_order)}] {each_column}: {best_n} states  ({elapsed:.0f}s, ETA {eta:.0f}s)')

print(f'\nDone in {time.time()-t0:.0f}s')
print(f'\nn_states distribution:')
ns_counter = Counter(n_states_dict.values())
for n in sorted(ns_counter.keys()):
    print(f'  {n} states: {ns_counter[n]} genes')

---
## Part D: Build SETIA input matrix

Rows = cell type (in sorted order, matching `sorted(Samples_Dic.keys())`)  
Cols = gene (in `ordered_genes` order)  
Values = state median (in log2 space if `USE_LOG2`)

In [ ]:
# Transpose TPM_matrix_T to get (cell_type × gene) matrix
# First row of TPM_matrix_T is the header (row indices), drop it for the actual matrix
header_row = TPM_matrix_T[0]
TPM_matrix = [list(row) for row in zip(*TPM_matrix_T[1:])]
std_matrix = [list(row) for row in zip(*std_matrix_T[1:])]

cell_types_sorted = sorted(Samples_Dic.keys())
setia_input = pd.DataFrame(TPM_matrix, index=cell_types_sorted, columns=ordered_genes)
setia_std   = pd.DataFrame(std_matrix, index=cell_types_sorted, columns=ordered_genes)
setia_input.index.name = 'cell_type'
setia_std.index.name = 'cell_type'

# Also build linear-space DataFrames from the same clusters (transformed during GMM call)
TPM_matrix_linear = [list(row) for row in zip(*TPM_matrix_T_linear[1:])]
std_matrix_linear = [list(row) for row in zip(*std_matrix_T_linear[1:])]
setia_input_linear_direct = pd.DataFrame(TPM_matrix_linear, index=cell_types_sorted, columns=ordered_genes)
setia_std_linear           = pd.DataFrame(std_matrix_linear, index=cell_types_sorted, columns=ordered_genes)
setia_input_linear_direct.index.name = 'cell_type'
setia_std_linear.index.name = 'cell_type'

print(f'SETIA input matrix: {setia_input.shape}')
print(f'\nValue stats (log2 CPM space):')
print(f'  Min: {setia_input.values.min():.3f}')
print(f'  Max: {setia_input.values.max():.3f}')
print(f'  Mean: {setia_input.values.mean():.3f}')
print(f'\nPreview (first 5 cell types × first 8 genes):')
print(setia_input.iloc[:5, :8].round(2))

### D1. Convert back to linear CPM if desired

If SETIA expects linear CPM (per million), uncomment the next cell to invert log2.

In [ ]:
if USE_LOG2:
    # Use DIRECT linear-space medians (from clusters, not inverted log2 medians)
    # to ensure medians and stds are in same space (both computed on the same linear cluster values)
    setia_input_linear = setia_input_linear_direct.copy()
    setia_input_linear[setia_input_linear < 0] = 0   # numeric safety  # numeric safety
    print('Linear CPM SETIA input (after inverting log2):')
    print(f'  Min: {setia_input_linear.values.min():.3f}')
    print(f'  Max: {setia_input_linear.values.max():.3f}')
    print(f'  Mean: {setia_input_linear.values.mean():.3f}')
    print()
    print('Preview (first 5 cell types × first 8 genes):')
    print(setia_input_linear.iloc[:5, :8].round(1))

### D2. Spot-check known markers

In [ ]:
spot_checks = [
    ('GATA1', '03_Late.Eryth'),
    ('HBB',   '03_Late.Eryth'),
    ('PAX5',  '16_Pre.B'),
    ('CD14',  '12_CD14.Mono.2'),
    ('CD3D',  '19_CD8.N'),
    ('TCF4',  '09_pDC'),
    ('CD34',  '01_HSC'),
    ('SPI1',  '11_CD14.Mono.1'),
]
print("SETIA input values (linear CPM) at known marker cell types:")
for gene, ct in spot_checks:
    if gene in setia_input.columns and ct in setia_input.index:
        v_log = setia_input.loc[ct, gene]
        v_lin = (2 ** v_log) - 1 if USE_LOG2 else v_log
        print(f'  {gene} in {ct}: log2={v_log:.2f}  (linear CPM ≈ {v_lin:.1f})')
    else:
        print(f'  {gene} or {ct} not found')

### D3. Save

In [ ]:
# Save log2 CPM version (matches yeast pipeline scale)
setia_input.to_csv(f"{HUMAN_BASE}/GTEx_v11/result/setia_input_log2_CPM.tsv", sep='\t', float_format='%.4f')
setia_std.to_csv(f"{HUMAN_BASE}/GTEx_v11/result/setia_input_log2_CPM_std.tsv", sep='\t', float_format='%.4f')
print('Saved: setia_input_log2_CPM.tsv')
print('Saved: setia_input_log2_CPM_std.tsv (state std dev per (cell_type, gene))')

# === NEW: save linear-space std (computed from same clusters as log2 stats) ===
setia_std_linear.to_csv(f"{HUMAN_BASE}/GTEx_v11/result/setia_input_linear_CPM_std.tsv", sep='\t', float_format='%.4f')
print('Saved: setia_input_linear_CPM_std.tsv (state std dev in LINEAR CPM space)')

# Save linear CPM version
if USE_LOG2:
    setia_input_linear.to_csv(f"{HUMAN_BASE}/GTEx_v11/result/setia_input_linear_CPM.tsv", sep='\t', float_format='%.4f')
    print('Saved: setia_input_linear_CPM.tsv')

# Save pseudocell info
info_df.to_csv(f"{HUMAN_BASE}/GTEx_v11/result/bmmc_pseudocell_info.tsv", sep='\t', index=False)
print('Saved: bmmc_pseudocell_info.tsv (pseudocell composition record)')

# Save n_states per gene
ns_df = pd.DataFrame([{'gene': g, 'n_states': n} for g, n in n_states_dict.items()])
ns_df.to_csv(f"{HUMAN_BASE}/GTEx_v11/result/bmmc_state_calling_n_states.tsv", sep='\t', index=False)
print('Saved: bmmc_state_calling_n_states.tsv')

# Yeast pipeline also writes Steady_state_count.txt (per-gene state assignments)
# That's already been written by select_and_convert_gmm_aicc inside ./result/
print('\nYeast-format state assignments per gene also saved to: ./result/Steady_state_count.txt')
print('GMM plots saved to: ./result/GMM_figures/AIC/ (one per gene)')

---
## Done

Outputs:
- `setia_input_linear_CPM.tsv` — main SETIA input (rows = cell type, cols = gene, values = state median in CPM)
- `setia_input_log2_CPM.tsv` — same in log2 space
- `setia_input_log2_CPM_std.tsv` — std deviation of state assignment per (cell_type, gene)
- `bmmc_pseudocell_info.tsv` — which cells went into which pseudocell
- `bmmc_state_calling_n_states.tsv` — number of states found per gene
- `./result/Steady_state_count.txt` — yeast-pipeline format state assignments
- `./result/GMM_figures/AIC/*.jpg` — visualization of state calling per gene

In [ ]:
# Data-driven candidate master regulator selection — BMMC version.
# Operates on n_states_dict (already built by your state-calling loop above)
# and setia_input_linear (cell type × gene matrix of state medians).

MIN_STATES        = 2   # gene must have multiple stable states across BMMC cell types
MAX_HIGH_TYPES    = 6   # gene's highest state appears in <= 6 of 26 cell types (tissue-specific)
MIN_DYNAMIC_RANGE = 1.0 # log2 high-state median vs low-state median >= 1 (clear separation)

candidates = []
for gene in ordered_genes:
    if n_states_dict.get(gene, 1) < MIN_STATES:
        continue
    
    # gene's expression across 26 cell types (log2 CPM space)
    vals = setia_input[gene].values
    # number of cell types in the highest state = those near the max value
    threshold_high = vals.max() - 0.5  # within 0.5 of peak counts as "high"
    n_high = int((vals >= threshold_high).sum())
    if not (1 <= n_high <= MAX_HIGH_TYPES):
        continue
    
    # dynamic range: peak vs mean of non-peak
    non_peak = vals[vals < threshold_high]
    if len(non_peak) == 0:
        continue
    dynamic_range = vals.max() - non_peak.mean()
    if dynamic_range < MIN_DYNAMIC_RANGE:
        continue
    
    # which cell types are in high state? (these are the candidate "target lineages")
    high_cts = [cell_types_sorted[i] for i, v in enumerate(vals) if v >= threshold_high]
    candidates.append({'gene': gene, 'n_states': n_states_dict[gene],
                       'n_high_types': n_high, 'dynamic_range': round(dynamic_range, 2),
                       'high_in': ', '.join(high_cts)})

cand_df = pd.DataFrame(candidates).sort_values('dynamic_range', ascending=False)
print(f'Data-driven candidates: {len(cand_df)} of {len(ordered_genes)} genes')
print(cand_df.to_string(index=False))

In [ ]:
print('\t'.join([str(i) for i in range(0, 56)]))